In [26]:
import os
import sys
import numpy as np
from sumo_rl import SumoEnvironment
from sumo_rl.agents import QLAgent
from sumo_rl.exploration import EpsilonGreedy
from gymnasium import spaces

from datetime import datetime

In [27]:
if "SUMO_HOME" in os.environ:
    tools = os.path.join(os.environ["SUMO_HOME"], "tools")
    sys.path.append(tools)
else:
    sys.exit("Please declare the environment variable 'SUMO_HOME'")


In [29]:
# -------------------------------
# PATH SETUP (CLEAN ✅)
# -------------------------------
CURRENT_DIR = r"c:\Users\VICTUS\Documents\AI_Project\Intelligent-Traffic-Signal-Control-using-Reinforcement-Learning\Q-Learning"
PROJECT_ROOT = os.path.abspath(os.path.join(CURRENT_DIR, ".."))
sys.path.insert(0, PROJECT_ROOT)

NET_FILE = os.path.join(CURRENT_DIR, "2way-single-intersection", "single-intersection.net.xml")
ROUTE_FILE = os.path.join(CURRENT_DIR, "2way-single-intersection", "single-intersection-vhvh.rou.xml")

In [30]:
def get_valid_phases(traffic_signal):
    logic = traci.trafficlight.getAllProgramLogics(traffic_signal.id)[0]
    phases = logic.phases
    return [i for i, p in enumerate(phases) if 'G' in p.state]

In [31]:
def safe_encode(env, state, ts_id):
    try:
        encoded = env.encode(state, ts_id)
        return tuple(encoded) if isinstance(encoded, (list, np.ndarray)) else encoded
    except (TypeError, IndexError):
        return tuple(np.round(state, 2)) if isinstance(state, (list, np.ndarray)) else str(state)


In [ ]:
def score_phase(traffic_signal, phase_index, queues):
    logic = traci.trafficlight.getAllProgramLogics(traffic_signal.id)[0]
    phase = logic.phases[phase_index]
    state = phase.state

    score = 0
    total_queue = sum(queues) if len(queues) > 0 else 1

    for signal in state:
        if signal == 'G':
            score += total_queue / len(state)

    return score

In [36]:
def choose_action(ts, agent, traffic_signal):
    queues = traffic_signal.get_lanes_queue()
    valid_phases = get_valid_phases(traffic_signal)

    # fallback safety
    if not valid_phases:
        return 0

    # -------------------
    # RL + HEURISTIC MIX
    # -------------------
    if np.random.rand() < 0.7:
        rl_action = agent.act()
        return valid_phases[rl_action % len(valid_phases)]
    else:
        return max(valid_phases, key=lambda p: score_phase(traffic_signal, p, queues))

In [37]:
# For notebook compatibility, define args as a simple object instead of using argparse
class Args:
    def __init__(self):
        self.route = ROUTE_FILE
        self.alpha = 0.3
        self.gamma = 0.99
        self.epsilon = 0.3
        self.min_epsilon = 0.005
        self.decay = 0.9995
        self.min_green = 10
        self.max_green = 30
        self.gui = False
        self.fixed = False
        self.seconds = 100000
        self.runs = 1

args = Args()

In [38]:
experiment_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

out_csv = os.path.join(
    CURRENT_DIR,
    "Outputs",
    "2way-single-intersection",
    f"{experiment_time}_alpha{args.alpha}_gamma{args.gamma}_eps{args.epsilon}"
)

os.makedirs(os.path.dirname(out_csv), exist_ok=True)

# -------------------------------
# ENV INIT (FIXED PATHS ✅)
# -------------------------------
env = SumoEnvironment(
    net_file=NET_FILE,
    route_file=args.route,
    out_csv_name=out_csv,
    use_gui=args.gui,   # ✅ FIXED (you wanted this)
    num_seconds=args.seconds,
    min_green=args.min_green,
    max_green=args.max_green,
    sumo_warnings=False,
)

# -------------------------------
# TRAINING LOOP
# -------------------------------
for run in range(1, args.runs + 1):
    print(f"\n🚀 Starting Run {run}/{args.runs}")

    initial_states = env.reset()
    ql_agents = {}

    for ts in env.ts_ids:
        starting_state = safe_encode(env, initial_states[ts], ts)

        valid_phases = get_valid_phases(env.traffic_signals[ts])

        ql_agents[ts] = QLAgent(
            starting_state=starting_state,
            state_space=env.observation_space,
            action_space=spaces.Discrete(len(valid_phases) if len(valid_phases) > 0 else 4),
            alpha=args.alpha,
            gamma=args.gamma,
            exploration_strategy=EpsilonGreedy(
                initial_epsilon=args.epsilon,
                min_epsilon=args.min_epsilon,
                decay=args.decay
            ),
        )

    done = {"__all__": False}
    step = 0

    while not done["__all__"]:
        actions = {}

        for ts in ql_agents.keys():
            traffic_signal = env.traffic_signals[ts]
            actions[ts] = choose_action(ts, ql_agents[ts], traffic_signal)

        s, r, done, _ = env.step(action=actions)

        for ts in ql_agents.keys():
            next_state = safe_encode(env, s[ts], ts)
            ql_agents[ts].learn(next_state=next_state, reward=r[ts])

        step += 1

        # 📊 PROGRESS TRACKING
        if step % 1000 == 0:
            print(f"Step: {step}")

    env.save_csv(out_csv, run)

env.close()


🚀 Starting Run 1/1


AttributeError: 'TrafficSignal' object has no attribute 'phases'

In [24]:
env = SumoEnvironment(
    net_file="2way-single-intersection\\single-intersection.net.xml",
    route_file=args.route,
    out_csv_name=out_csv,
    use_gui=False,
    num_seconds=args.seconds,
    min_green=args.min_green,
    max_green=args.max_green,
    sumo_warnings=False,
)

In [25]:
for run in range(1, args.runs + 1):
    print(f"\n🚀 Starting Run {run}/{args.runs}")

    initial_states = env.reset()

    ql_agents = {}

    for ts in env.ts_ids:
        starting_state = safe_encode(env, initial_states[ts], ts)

        # Determine valid phases dynamically
        valid_phases = get_valid_phases(env.traffic_signals[ts])

        ql_agents[ts] = QLAgent(
            starting_state=starting_state,
            state_space=env.observation_space,
            action_space=len(valid_phases) if len(valid_phases) > 0 else 4,
            alpha=args.alpha,
            gamma=args.gamma,
            exploration_strategy=EpsilonGreedy(
                initial_epsilon=args.epsilon,
                min_epsilon=args.min_epsilon,
                decay=args.decay
            ),
        )


🚀 Starting Run 1/1


AttributeError: 'int' object has no attribute 'n'